In [2]:
pip install PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 6.1 MB/s eta 0:00:00


In [7]:
from pathlib import Path
import pandas as pd
import zipfile
from PyPDF2 import PdfReader




def extract_zip(zip_path: str, extract_folder: str):
    """Extract zip file."""

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_folder)

    print("ZIP extracted successfully")



def extract_text_from_pdf(pdf_path: Path) -> str:
    """Extract text from one PDF file."""

    reader = PdfReader(str(pdf_path))

    pages_text = [
        page.extract_text() or ""
        for page in reader.pages
    ]

    return "\n".join(pages_text).strip()



def read_pdf_folder(folder_path: str) -> pd.DataFrame:
    """Read all PDF files from folder."""

    folder = Path(folder_path)

    if not folder.exists():
        raise FileNotFoundError(
            f"Folder not found: {folder_path}"
        )

    records = []

    # rglob searches PDFs inside subfolders too
    for pdf_file in folder.rglob("*.pdf"):

        try:

            records.append({
                "filename": pdf_file.name,
                "text": extract_text_from_pdf(pdf_file)
            })

        except Exception as e:

            records.append({
                "filename": pdf_file.name,
                "text": "",
                "error": str(e)
            })

    return pd.DataFrame(records)



def save_to_csv(
    df: pd.DataFrame,
    output_path: str
) -> None:

    df.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"Saved successfully: {output_path}"
    )


def main():

    # ZIP file path
    zip_path = "/content/Education.zip"

    # Extraction folder
    extract_folder = "/content/Education"

    # Output CSV
    output_path = "/content/pdf_text_output.csv"

    # Extract ZIP
    extract_zip(
        zip_path,
        extract_folder
    )

    # Read PDFs
    df = read_pdf_folder(
        extract_folder
    )

    print(
        "Total PDFs processed:",
        len(df)
    )

    print(df.head())

    # Save CSV
    save_to_csv(
        df,
        output_path
    )

    return df



if __name__ == "__main__":

    df = main()

ZIP extracted successfully
Total PDFs processed: 5000
                            filename  \
0  Resume_033338_Bradley_Richard.pdf   
1     Resume_030995_Chloe_Nelson.pdf   
2  Resume_033039_Amanda_Espinoza.pdf   
3        Resume_032251_Mary_Webb.pdf   
4    Resume_034171_William_White.pdf   

                                                text  
0  Bradley Richard\n bradley.richard33338@poole.c...  
1  Chloe Nelson\n chloe.nelson30995@saunders-bray...  
2  Amanda Espinoza\n amanda.espinoza33039@jones-p...  
3  Mary Webb\n mary.webb32251@ross.info | 7725536...  
4  William White\n william.white34171@griffin-dav...  
Saved successfully: /content/pdf_text_output.csv


In [12]:
import pandas as pd
from PyPDF2 import PdfReader

pdf_path = "20_Job_Descriptions (2).pdf"

reader = PdfReader(pdf_path)

data = []

for page in reader.pages:

    page_text = page.extract_text() or ""

    lines = [line.strip() for line in page_text.split("\n") if line.strip()]

    # First line = Job Title
    name = lines[0]

    # Remaining text = Description
    text = "\n".join(lines[1:])

    data.append({
        "name": name,
        "text": text
    })

df = pd.DataFrame(data)

print(df.shape)   # (20, 2)
print(df.head())

# Save if needed
df.to_csv("job_descriptions.csv", index=False)

(20, 2)
                               name  \
0  20 Professional Job Descriptions   
1             Azure DevOps Engineer   
2                    Data Scientist   
3                      Data Analyst   
4              Full Stack Developer   

                                                text  
0  Azure Data Engineer\nJob Summary\nWe are seeki...  
1  Job Summary\nWe are seeking a skilled Azure De...  
2  Job Summary\nWe are seeking a skilled Data Sci...  
3  Job Summary\nWe are seeking a skilled Data Ana...  
4  Job Summary\nWe are seeking a skilled Full Sta...  


In [13]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

# ==============================
# 1. Load CSV files
# ==============================

job_df = pd.read_csv("job_descriptions.csv")
resume_df = pd.read_csv("resumes.csv")

print(job_df.shape)
print(resume_df.shape)

# ==============================
# 2. Clean missing values
# ==============================

job_df["text"] = job_df["text"].fillna("")
resume_df["text"] = resume_df["text"].fillna("")

# ==============================
# 3. Load Hugging Face embedding model
# ==============================

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# ==============================
# 4. Generate embeddings
# ==============================

job_embeddings = model.encode(
    job_df["text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

resume_embeddings = model.encode(
    resume_df["text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Job Embeddings Shape:", job_embeddings.shape)
print("Resume Embeddings Shape:", resume_embeddings.shape)

# ==============================
# 5. Save embeddings as NPY files
# ==============================

np.save("job_embeddings.npy", job_embeddings)
np.save("resume_embeddings.npy", resume_embeddings)

print("Saved job_embeddings.npy")
print("Saved resume_embeddings.npy")

# ==============================
# 6. Optional: Save updated CSV also
# ==============================

job_df.to_csv("job_descriptions_clean.csv", index=False)
resume_df.to_csv("resumes_clean.csv", index=False)

print("Completed successfully")


import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances

# ==============================
# 1. Load CSV files
# ==============================

job_df = pd.read_csv("job_descriptions.csv")
resume_df = pd.read_csv("resumes.csv")

# ==============================
# 2. Load NPY embeddings
# ==============================

job_embeddings = np.load("job_embeddings.npy")
resume_embeddings = np.load("resume_embeddings.npy")

print("Job Embeddings:", job_embeddings.shape)
print("Resume Embeddings:", resume_embeddings.shape)

# ==============================
# 3. Calculate Similarity / Distance
# ==============================

cosine_scores = cosine_similarity(job_embeddings, resume_embeddings)

euclidean_scores = euclidean_distances(job_embeddings, resume_embeddings)

(20, 2)
(5000, 2)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Job Embeddings Shape: (20, 384)
Resume Embeddings Shape: (5000, 384)
Saved job_embeddings.npy
Saved resume_embeddings.npy
Completed successfully
Job Embeddings: (20, 384)
Resume Embeddings: (5000, 384)
